# RAG 구축과 성능 평가 — 명지대학교 규정 검색 챗봇 만들기

> **이 실습에서 할 일 (한 줄 요약)**
> 명지대학교 규정집 PDF 239개를 읽어서, 질문하면 **규정을 찾아보고 근거와 함께 답하는 챗봇(RAG)** 을 만들고, 그 챗봇이 **얼마나 잘하는지 점수로 채점**합니다.

### 전체 흐름

```
[Part A. 만들기]
 규정 PDF → ① 읽고 정리 → ② 잘게 자르기(청크) → ③ 숫자로 바꾸기(임베딩) → ④ 벡터 DB에 저장
 질문 → ⑤ 질문도 숫자로 → ⑥ 비슷한 청크 찾기 → ⑦ LLM이 청크를 보고 답변

[Part B. 채점하기]
 채점용 문제지(Golden Set) → 검색 점수(잘 찾았나?) + 답변 점수(잘 답했나?) → 문제 원인 찾기 → 리포트
```

### 꼭 알아야 할 용어

| 용어 | 쉬운 뜻 |
|---|---|
| RAG | **R**etrieval-**A**ugmented **G**eneration. 먼저 자료를 **찾아보고(검색)** 그걸 바탕으로 **답을 만드는(생성)** 방식 |
| 청크(Chunk) | 긴 문서를 검색하기 좋게 자른 **조각** (이 실습에서는 약 500자) |
| 임베딩(Embedding) | 글의 **의미를 숫자 목록(벡터)** 으로 바꾼 것. 뜻이 비슷한 글은 숫자도 비슷함 |
| 벡터 DB | 임베딩을 저장해 두고 **비슷한 것을 빨리 찾아주는** 데이터베이스 (여기서는 Milvus Lite) |
| Retriever(검색기) | 질문을 받으면 관련 청크를 찾아오는 **LangChain 부품** |
| LLM | 답변을 만들어 주는 언어 모델 (Upstage `solar-mini4`) |
| Golden Set | 채점용 **문제지**: 질문 + 모범답안 + 정답 근거 |

### 사용 도구
- LangChain: `PyPDFLoader`(PDF 읽기), `RecursiveCharacterTextSplitter`(자르기), `HuggingFaceEmbeddings`(임베딩), `Milvus`(벡터 DB), LCEL 체인(부품 연결)
- 임베딩 모델: `dragonkue/BGE-m3-ko` / 답변 LLM: `solar-mini4` / 채점 LLM: `solar-pro4`

### 실행 전 준비 (Colab)
1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택
2. Upstage API Key 발급: https://console.upstage.ai/api-keys (왼쪽 🔑 보안 비밀에 `UPSTAGE_API_KEY`로 등록하면 편리)
3. 위에서부터 **셀을 차례대로 실행** (Shift + Enter). 규정 데이터는 자동으로 내려받음
4. PDF 읽기 결과는 `/content/raw_docs.json`에 저장 → 같은 세션에서 다시 실행하면 빨라짐

---
## 0. 실행 준비

### 0-1. 구글 드라이브 연결 + 라이브러리 설치
- 임베딩 모델(약 2GB)을 드라이브에 저장해 두면 다음 실행부터 다시 받지 않음
- `GPU: Tesla T4`가 나오면 정상. `없음`이 나오면 런타임 유형을 T4 GPU로 바꾸고 다시 실행

In [1]:
from google.colab import drive
drive.mount('/content/drive')                                   # 구글 드라이브 연결 (권한 허용 창이 뜸)

import os, torch

os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'      # 임베딩 모델을 드라이브에 저장(캐시)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 → 런타임 유형을 T4 GPU로 변경")

# 필요한 라이브러리 설치 (-q: 설치 메시지를 짧게)
!pip install -q langchain langchain-community langchain-huggingface langchain-upstage langchain-milvus "pymilvus[milvus_lite]" pypdf sentence-transformers pandas

Mounted at /content/drive
GPU: Tesla T4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.5 MB/s eta 0:00:00


### 0-2. 규정 데이터 내려받기 + API Key 설정
- GitHub에 올려 둔 `명지대규정.zip`(약 21MB)을 내려받아 압축을 풂
- 결과: 규정 파일 240개, 편(제1편~제6편, 부록) 폴더 7개

In [2]:
from getpass import getpass
from google.colab import userdata
import zipfile, urllib.request

# 파일 위치를 변수로 정해 두기
DATA_URL = 'https://github.com/bjkim2004/Vector_Graph/raw/main/%EB%AA%85%EC%A7%80%EB%8C%80%EA%B7%9C%EC%A0%95.zip'
ZIP_PATH = '/content/명지대규정.zip'
DATA_DIR = '/content/명지대규정'                     # 규정 PDF가 들어갈 폴더
CACHE_PATH = '/content/raw_docs.json'              # PDF를 읽은 결과를 저장할 파일
DB_URI = '/content/mju_rag.db'                     # 벡터 DB 파일 (Milvus Lite는 파일 하나로 동작)

# 폴더가 없을 때만 내려받기 → 압축 풀기
if not os.path.isdir(DATA_DIR):
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)
print("규정 파일 수:", sum(len(files) for _, _, files in os.walk(DATA_DIR)))
print("편 폴더:", sorted(os.listdir(DATA_DIR)))

# API Key: 보안 비밀(🔑)에 있으면 가져오고, 없으면 직접 입력
try:
    os.environ['UPSTAGE_API_KEY'] = userdata.get('UPSTAGE_API_KEY')
except Exception:
    os.environ['UPSTAGE_API_KEY'] = getpass('UPSTAGE_API_KEY: ')

규정 파일 수: 240
편 폴더: ['부록,참고규정', '제1편 명지학원', '제2편 학칙,시행규칙', '제3편 학사행정', '제4편 일반행정', '제5편 부속,부설교육,연구소,위원회', '제6편 산학협력단']
UPSTAGE_API_KEY: ··········


### 0-3. 모델 준비 (임베딩 모델 1개 + LLM 2개)
- `embeddings`: 글 → 숫자 벡터(1024개 숫자)로 바꾸는 모델
- `llm`: 답변을 만드는 모델 / `judge`: 답변을 **채점**하는 모델
- 답변 모델과 채점 모델을 다르게 쓰는 이유: 자기가 쓴 답에 후한 점수를 주는 **자기 선호 편향**을 줄이기 위해

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_upstage import ChatUpstage

# 임베딩 모델: 한국어에 강한 BGE-m3-ko
embeddings = HuggingFaceEmbeddings(
    model_name="dragonkue/BGE-m3-ko",
    encode_kwargs={"normalize_embeddings": True,   # 벡터 길이를 1로 맞춤 → 코사인 유사도 계산이 쉬워짐
                   "batch_size": 32})               # 한 번에 32개씩 처리

llm = ChatUpstage(model="solar-mini4", temperature=0)    # 답변 모델 (temperature=0: 매번 같은 답)
judge = ChatUpstage(model="solar-pro4", temperature=0)   # 채점 모델 (답변 모델과 분리)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


---
# Part A. RAG 만들기

## LLM 혼자서는 왜 부족할까?
- LLM은 **요리 실력은 좋지만 우리 집 냉장고에 뭐가 있는지 모르는 요리사**와 같음
- 학교 규정처럼 **학습한 적 없거나 최근에 바뀐 정보**는 모름 → 그럴듯하게 틀린 답(환각)을 만듦
- 실제 규정: 인공지능·소프트웨어융합대학 졸업학점 **134학점** (학칙 제50조, 2025.4.1. 개정)
- 아래 셀에서 LLM에게 **규정을 보여주지 않고** 그냥 물어봄 → 몇 학점이라고 답하는지 확인

In [4]:
q = "명지대학교 인공지능·소프트웨어융합대학 학생은 졸업하려면 몇 학점을 취득해야 하나요?"
print(llm.invoke(q).content)      # 규정 없이 LLM에게 바로 질문

명지대학교 인공지능·소프트웨어융합대학 학생의 졸업 요건은 **전공·교양·교양선택·교양필수 등 전체 이수 학점을 합산하여 130학점 이상**을 취득해야 합니다.

다만, 구체적인 졸업 요건은 **입학 연도, 학번, 전공(인공지능·소프트웨어융합대학 내 세부 전공), 복수전공/부전공 여부, 교직 이수 여부** 등에 따라 달라질 수 있습니다. 예를 들어, 일부 학과는 졸업에 필요한 전공 학점이 별도로 정해져 있거나, 교양 필수 과목 이수 조건이 다를 수 있습니다.

따라서 정확한 졸업 학점 기준은 **명지대학교 학과 홈페이지의 졸업 요건 안내** 또는 **학과 사무실/교무팀**에 확인하시는 것이 가장 정확합니다.


> **결과 보기**: LLM이 **130학점**이라고 자신 있게 답함 → 실제(134학점)와 다름. 이것이 **환각**이며, RAG로 해결하려는 문제

---
## [1단계] 데이터 준비 — ① 수집 및 전처리

- `PyPDFLoader`: PDF를 **페이지마다** 읽어 `Document`(글 + 메타데이터)로 만듦 → 규정 하나의 페이지들을 이어 붙여 **규정 1개 = 문서 1개**로 만듦
- **메타데이터**(글에 붙이는 이름표): 폴더 이름과 파일 이름에서 뽑아냄
  - 편(part) / 장(chapter) / 규정번호(reg_id) / 규정명(title)
  - 예: `제2편 학칙,시행규칙/제1장 학칙/2-1-1 명지대학교 학칙.pdf` → reg_id=`2-1-1`, title=`명지대학교 학칙`
- hwp 파일 1개는 PDF 로더로 읽을 수 없어 제외

In [5]:
import re, glob
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

def parse_meta(path):
    """파일 경로에서 메타데이터(편·장·규정번호·규정명)를 뽑아내는 함수"""
    p = Path(path)
    parts = p.relative_to(DATA_DIR).parts           # 예: ('제2편 학칙,시행규칙', '제1장 학칙', '2-1-1 명지대학교 학칙.pdf')
    m = re.match(r"^([\d\-]+)\.?\s*(.+)$", p.stem)  # 파일명 '2-1-1 명지대학교 학칙'을 번호와 이름으로 나눔
    reg_id, title = (m.group(1), m.group(2).strip()) if m else (p.stem, p.stem)   # 번호가 없으면 파일명 그대로
    return {"part": parts[0],                        # 편
            "chapter": parts[1] if len(parts) > 2 else "",   # 장 (없으면 빈칸)
            "reg_id": reg_id, "title": title, "source": p.name}

pdf_paths = sorted(glob.glob(f"{DATA_DIR}/**/*.pdf", recursive=True))   # 모든 하위 폴더의 PDF 찾기
print("PDF 수:", len(pdf_paths))
print("제외(hwp):", [Path(p).name for p in glob.glob(f"{DATA_DIR}/**/*.hwp", recursive=True)])
print(parse_meta(pdf_paths[0]))                      # 첫 번째 파일의 메타데이터 확인

PDF 수: 239
제외(hwp): ['5-4-32  인권 · 감사위원회 규정.hwp']
{'part': '부록,참고규정', 'chapter': '', 'reg_id': '부록1. 교직원친화회 회칙', 'title': '부록1. 교직원친화회 회칙', 'source': '부록1. 교직원친화회 회칙.pdf'}


/tmp/ipykernel_1574/3680371680.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### PDF 239개 읽기
- 처음 실행할 때는 수 분이 걸림 → 결과를 `raw_docs.json`에 저장
- 다시 실행하면 저장된 파일을 바로 읽음(캐시)

In [6]:
import json
from tqdm.auto import tqdm                           # 진행률 막대 표시

if os.path.exists(CACHE_PATH):                       # 저장된 결과가 있으면 → 그걸 읽음
    raw_docs = [Document(**d) for d in json.load(open(CACHE_PATH, encoding="utf-8"))]
else:                                                # 없으면 → PDF를 하나씩 읽음
    raw_docs = []
    for path in tqdm(pdf_paths):
        pages = PyPDFLoader(path).load()                         # 페이지 목록
        text = "\n".join(p.page_content for p in pages)          # 페이지 글을 하나로 이어 붙임
        raw_docs.append(Document(page_content=text, metadata=parse_meta(path)))
    # 다음 실행을 위해 저장
    json.dump([{"page_content": d.page_content, "metadata": d.metadata} for d in raw_docs],
              open(CACHE_PATH, "w", encoding="utf-8"), ensure_ascii=False)

print("문서 수:", len(raw_docs), "/ 전체 글자 수:", sum(len(d.page_content) for d in raw_docs))

  0%|          | 0/239 [00:00<?, ?it/s]

문서 수: 239 / 전체 글자 수: 1737869


### 텍스트 정제 (청소)
- 페이지 번호(`- 5 -`) 같은 **쓸모없는 글자(노이즈)** 를 지우고, 줄바꿈·연속 공백을 정리
- 규정의 기본 단위인 **조(條)**, 예를 들어 `제50조(…)` 앞에 빈 줄(`\n\n`)을 넣음 → 다음 단계에서 **조 단위로 자르는 기준**이 됨

In [7]:
def clean(text):
    text = re.sub(r"-\s+\d{1,3}\s+-", " ", text)                            # ① 페이지 번호 '- 5 -' 지우기
    text = re.sub(r"\s+", " ", text)                                         # ② 줄바꿈·여러 칸 공백 → 공백 한 칸
    text = re.sub(r"\s*(제\s?\d+\s?조(?:의\s?\d+)?\s?\()", r"\n\n\1", text)  # ③ '제N조(' 앞에 빈 줄 넣기
    return text.strip()

docs = [Document(page_content=clean(d.page_content), metadata=d.metadata) for d in raw_docs]

# 예시: 명지대학교 학칙(2-1-1)이 어떻게 정리됐는지 앞부분 600자 확인
hakchik = next(d for d in docs if d.metadata["reg_id"] == "2-1-1")
print(hakchik.metadata)
print(hakchik.page_content[:600])

{'part': '제2편 학칙,시행규칙', 'chapter': '제1장 학칙', 'reg_id': '2-1-1', 'title': '명지대학교 학칙', 'source': '2-1-1 명지대학교 학칙.pdf'}
명지대학교 학칙 (최종개정일 2026.9.1.) 제 1 장 총 칙

제1조(목적) 이 학칙은 명지대학교(이하 "우리 대학교"라 한다)의 교육목적을 설정하 고 이를 달성하기 위한 교육조직, 학사운영, 교육과정 등에 관한 사항을 규정함을 목적으로 한다.

제2조(교육목적) 우리 대학교는 학교법인 명지학원의 설립정신인 기독교 정신을 바 탕으로 학술의 심오한 이론과 그 올바른 활용방법을 연구ㆍ교수하며 국가발전과 민족문화 창달을 통하여 인류사회 평화와 발전에 기여 봉사할 인재양성을 목적으 로 한다. 제 2 장 편 제 제 1 절 교육조직

제3조(교육조직) ①우리 대학교는 인문캠퍼스 인문대학, 사회과학대학, 경영대학, 미 디어ž휴먼라이프대학, 인공지능ž소프트웨어융합대학, 미래융합대학을 두고 자연캠 퍼스에 화학ž생명과학대학, 스마트시스템공과대학, 반도체žICT대학, 스포츠ž예술대 학, 건축대학, 글로벌학부를 두며, 양 캠퍼스에 아너칼리지(전공자유대학)와 방목 기초교육대학을 둔다.(변경 2004.3.1., 2004.3.2., 2005.8.1., 2008.7.1., 2010.3.1., 2012. 7.1., 2015.12.1., 2016.7.1., 2017.5.1., 2017


- 편(part)별로 규정이 몇 개이고 글자가 얼마나 되는지 표로 확인

In [8]:
import pandas as pd

stat = pd.DataFrame([{**d.metadata, "chars": len(d.page_content)} for d in docs])   # 문서마다 메타데이터 + 글자 수
stat.groupby("part")["chars"].agg(["count", "sum"]).rename(columns={"count": "규정 수", "sum": "글자 수"})

,규정 수,글자 수
part,,
"부록,참고규정",2,10379
제1편 명지학원,13,175134
"제2편 학칙,시행규칙",5,227521
제3편 학사행정,60,360035
제4편 일반행정,79,627197
"제5편 부속,부설교육,연구소,위원회",69,168351
제6편 산학협력단,11,70282


---
## [1단계] ② 데이터 청킹 (잘게 자르기)

- 왜 자르나? 규정 하나는 수만 자 → 통째로 검색하면 **어느 부분이 질문과 관련 있는지** 알 수 없음
- **고정 크기 청킹**: 글자 수(300자)로만 뚝뚝 자름 → 조문 **중간이 잘림**
- **조(條) 단위 청킹**: 빈 줄(조 경계)에서 먼저 자르고, 짧은 조는 300자 안에서 이어 붙임 → 조문이 온전히 유지
- 아래 결과에서 두 방식의 **잘린 위치**를 비교해 보기

In [9]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

# 고정 크기: 구분자 없이 300자마다 자름
fixed = CharacterTextSplitter(separator="", chunk_size=300, chunk_overlap=0)
# 조 단위: 빈 줄(조 경계) → 문장 끝('. ') → 공백 순서로 자를 곳을 찾음
article = RecursiveCharacterTextSplitter(separators=["\n\n", ". ", " ", ""], chunk_size=300, chunk_overlap=0,
                                         keep_separator="end")

sample = hakchik.page_content[hakchik.page_content.find("제47조("):][:700]   # 학칙 제47조부터 700자
print("[고정 크기]")
for c in fixed.split_text(sample): print(" -", c[:80].replace("\n", " "), "...")
print("[조 단위]")
for c in article.split_text(sample): print(" -", c[:80].replace("\n", " "), "...")

[고정 크기]
 - 제47조(학사과정의 학사경고) ①당해학기 성적이 평균평점 1.50미만인 자에게는 학 사경고를 하며, 이에 관한 세부사항은 총장이 따로 정한다.  ...
 - (교과목 이수) 학군무관후보생과정 학생은 매학기 개설되는 소정의 교과목 을 이수하여야 한다.  제47조의4(예비역 교관) 국방부장관이 추천하는  ...
 - 3.4, 2006.4.17, 2009.3.1, 2009.7.1) ②각 대학원과정의 수료에 필요한 이수학점은 석사과정 24학점 이상, 박사과정 3 ...
[조 단위]
 - 제47조(학사과정의 학사경고) ①당해학기 성적이 평균평점 1.50미만인 자에게는 학 사경고를 하며, 이에 관한 세부사항은 총장이 따로 정한다.  ...
 - 제47조의3(교과목 이수) 학군무관후보생과정 학생은 매학기 개설되는 소정의 교과목 을 이수하여야 한다.  제47조의4(예비역 교관) 국방부장관이 ...
 - 제48조(수료학점) ①학사과정의 수료학점에 관한 세부사항은 학칙시행규칙(학사과정) 에 의한다.(변경 2005.3.4, 2006.4.17, 200 ...


### 규정 문서에 맞춘 최적 청킹 (4가지 개선)
1. **개정 이력 주석 지우기**: `(변경 2005.3.1, 2006.4.17)`, `[본 조 신설 2017.9.1.]` 같은 이력은 질문과 상관없는 노이즈 → 검색 품질을 떨어뜨림
   - `[별표 1]`처럼 날짜·개정 표현이 없는 괄호는 남김
2. **조(條) 경계에서 먼저 자르기**: 빈 줄 → 문장 → 공백 순
3. **조 머리말 반복**: 긴 조가 여러 조각으로 나뉘면 뒤 조각에도 `제N조(제목) …`을 붙여 **어느 조항인지** 알 수 있게 함
4. **규정명 붙이기**: 조각 앞에 `[규정명]`을 붙이고, 조 번호를 `article` 메타데이터로 저장
- Part B에서 이 방식이 실제로 더 좋은지 점수로 비교함

**먼저 ①(주석 지우기)부터 확인** — 정규표현식(글자 패턴 찾기 규칙)으로 `(…변경/신설/삭제… 날짜)` 모양의 괄호를 찾아서 지움

In [10]:
import re

# 지울 주석의 모양: 괄호 ( ) 또는 [ ] 안에 '변경·신설·삭제·개정·제정·시행·이동' 중 하나 + 날짜나 숫자가 있음
NOTE = re.compile(
    r"\s?[(\[]"                                                # 여는 괄호 ( 또는 [
    r"[^()\[\]]*(?:변경|신설|삭제|개정|제정|시행|이동)"          # 키워드 중 하나
    r"[^()\[\]]*(?:\d{2,4}\s?\.\s?\d{1,2}\s?\.\s?\d{1,2}|\d{2,4}\s*\.\s*|\d+)" # 날짜(2005.3.1) 또는 숫자
    r"[^()\[\]]*[)\]]"                                         # 닫는 괄호 ) 또는 ]
)

# 조 머리말의 모양: 제50조(학사과정의 졸업), 제47조의3(교과목 이수) 등
ARTICLE_HEAD = re.compile(r"제\s?\d+\s?조(?:의\s?\d+)?\s?\([^)]*\)")

def strip_notes(text):
    return NOTE.sub("", text)        # 주석 모양을 찾아서 빈 글자로 바꿈(=삭제)

sample = hakchik.page_content[hakchik.page_content.find("제50조("):]
sample_cut = sample[:500]            # 학칙 제50조부터 500자만 확인

print("[주석 제거 전]")
print(sample_cut)
print("\n[주석 제거 후]")
print(strip_notes(sample_cut))

[주석 제거 전]
제50조(학사과정의 졸업) ①소정의 교육과정을 이수하고, 128학점(스마트시스템공과 대학, 반도체žICT대학, 인공지능ž소프트웨어융합대학은 134학점, 건축대학 건축학 부 건축학전공, 공간디자인학과는 164학점, 건축학부 전통건축전공은 170학점, 계 약학과는 120학점) 이상 취득한 자에 대하여 졸업을 인정한다. 다만, 본인의 의사 에 따라 졸업을 유예할 수 있다.(변경 2005.3.4, 2006.4.17, 2009.3.1, 2011.5.1, 2012.3.1., 2016.3.1., 2018.7.1., 2019.1.1., 2025.4.1.) ②(삭제 2009.3.1) ③전체 성적이 평균평점 4.0 이상(학․석사 연계과정인 자는 평균평점 3.5이상)이고, 소정 의 졸업학점을 취득한 자는 한 학기 조기졸업을 할 수 있다.(변경 2009.3.1., 2019.9.1.) ④학사과정의 졸업에 관한 세부사항은 학칙시행규칙(학사과정)에 의한다.(변경 2004.4.1.)

제51조(각 대학원과정의 

[주석 제거 후]
제50조(학사과정의 졸업) ①소정의 교육과정을 이수하고, 128학점(스마트시스템공과 대학, 반도체žICT대학, 인공지능ž소프트웨어융합대학은 134학점, 건축대학 건축학 부 건축학전공, 공간디자인학과는 164학점, 건축학부 전통건축전공은 170학점, 계 약학과는 120학점) 이상 취득한 자에 대하여 졸업을 인정한다. 다만, 본인의 의사 에 따라 졸업을 유예할 수 있다. ② ③전체 성적이 평균평점 4.0 이상(학․석사 연계과정인 자는 평균평점 3.5이상)이고, 소정 의 졸업학점을 취득한 자는 한 학기 조기졸업을 할 수 있다. ④학사과정의 졸업에 관한 세부사항은 학칙시행규칙(학사과정)에 의한다.

제51조(각 대학원과정의 


### 최적 청킹 함수 `make_chunks`
- 입력: 정리된 문서 목록 / 출력: 청크 목록
- 청크마다 `chunk_id`(고유 번호)를 붙임 → 나중에 **어떤 청크가 검색됐는지** 추적할 때 사용
- 결과: 청크 **4,150개**. 예시로 학칙 제50조 청크를 출력

In [11]:
def make_chunks(docs, size=500, overlap=50, prefix=None):
    # 자르는 도구: 조 경계(\n\n) → 문장('. ') → 공백 순서로 자를 곳을 찾음
    splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", ". ", " ", ""], chunk_size=size, chunk_overlap=overlap,
        keep_separator="end", add_start_index=True)                   # 청크가 원문 몇 번째 글자에서 시작하는지 기록
    chunks = []
    for d in docs:
        text = strip_notes(d.page_content)                            # ① 개정 이력 주석 지우기
        heads = [(m.start(), m.group()) for m in ARTICLE_HEAD.finditer(text)]   # 조 머리말 위치 목록
        for c in splitter.split_documents([Document(page_content=text, metadata=d.metadata)]):   # ② 조 경계 우선 자르기
            start = c.metadata.pop("start_index")                     # 이 청크의 시작 위치
            head = next((h for p, h in reversed(heads) if p <= start), "")   # 시작 위치 바로 앞의 조 = 이 청크가 속한 조
            body = c.page_content.strip()
            if head and not body.startswith(head.split("(")[0]):
                body = f"{head} … {body}"                             # ③ 조 머리말이 없으면 앞에 붙여 줌
            c.page_content = f"[{c.metadata['title']}] {body}"        # ④ 규정명 붙이기
            c.metadata["article"] = head.split("(")[0].replace(" ", "")   # 조 번호(예: 제50조)를 메타데이터로
            chunks.append(c)
    for i, c in enumerate(chunks):
        c.metadata["chunk_id"] = f"{prefix or size}-{i}"              # 고유 번호 붙이기 (예: 500-508)
    return chunks

chunks = make_chunks(docs, size=500)
print("청크 수:", len(chunks))
example = next(c for c in chunks if c.metadata["reg_id"] == "2-1-1" and c.metadata["article"] == "제50조")
print(example.metadata)
print(example.page_content)

청크 수: 4150
{'part': '제2편 학칙,시행규칙', 'chapter': '제1장 학칙', 'reg_id': '2-1-1', 'title': '명지대학교 학칙', 'source': '2-1-1 명지대학교 학칙.pdf', 'article': '제50조', 'chunk_id': '500-508'}
[명지대학교 학칙] 제50조(학사과정의 졸업) ①소정의 교육과정을 이수하고, 128학점(스마트시스템공과 대학, 반도체žICT대학, 인공지능ž소프트웨어융합대학은 134학점, 건축대학 건축학 부 건축학전공, 공간디자인학과는 164학점, 건축학부 전통건축전공은 170학점, 계 약학과는 120학점) 이상 취득한 자에 대하여 졸업을 인정한다. 다만, 본인의 의사 에 따라 졸업을 유예할 수 있다. ② ③전체 성적이 평균평점 4.0 이상(학․석사 연계과정인 자는 평균평점 3.5이상)이고, 소정 의 졸업학점을 취득한 자는 한 학기 조기졸업을 할 수 있다. ④학사과정의 졸업에 관한 세부사항은 학칙시행규칙(학사과정)에 의한다.


- ③ 조 머리말 반복이 적용된 청크 예시 (`제N조(…) … 본문` 모양)

In [12]:
# 긴 조가 나뉜 경우: 뒤쪽 청크에도 조 머리말을 붙임
for c in [c for c in chunks if " … " in c.page_content][:3]:
    print(c.metadata["article"], "|", c.page_content[:120])

제25조 | [부록1. 교직원친화회 회칙] 제25조(예․결산 보고) … 부 칙 ①(시행일) 이 개정 회칙은 2000년 3월 1일부터 시행한다. ②(퇴직위로금에 관한 경과조치) 퇴직위로금의 지급은 제23조의 2 "별표 1의 5호"
제25조 | [부록1. 교직원친화회 회칙] 제25조(예․결산 보고) … 다만, 종전의 회칙에 따라 대여원금이 회수되지 아니한 자 는 연장을 신청하여야 한다. 부 칙 ①(시행일) 이 개정 회칙은 2011년 4월 1일부터 시행한다.
제25조 | [부록1. 교직원친화회 회칙] 제25조(예․결산 보고) … (별표 1) 제 경조금의 종류와 금액 제 경조금의 종류와 금액 1. 축 의 금 가. 회원결혼 : 100,000원 나. 다. 라. 자녀결혼 : 100,000원


---
## [1단계] ③ 임베딩 및 ④ 데이터 저장

- **임베딩**: 글 → 1024개 숫자. 뜻이 비슷한 글은 숫자 목록도 비슷함
- **벡터 DB 저장**: `Milvus.from_documents` 한 줄로 [청크 임베딩 → 컬렉션(테이블) 만들기 → 원문·메타데이터·벡터 저장]을 한 번에 처리
- 비슷한 정도는 **코사인 유사도**로 계산 (1에 가까울수록 비슷). `FLAT` = 모든 청크와 하나하나 비교(정확함)

In [13]:
v = embeddings.embed_query("휴학")     # '휴학'이라는 글자를 벡터로 바꿔 보기
print("벡터 차원:", len(v), "/ 앞 5개:", [round(x, 3) for x in v[:5]])

벡터 차원: 1024 / 앞 5개: [-0.055, 0.038, -0.001, -0.007, -0.01]


In [14]:
import logging
from langchain_milvus import Milvus

# Milvus Lite가 남기는 불필요한 오류 로그 숨기기 (저장·검색에는 영향 없음)
logging.getLogger("grpc._server").setLevel(logging.CRITICAL)

def build_store(chunks, name):
    """청크 목록을 임베딩해서 Milvus 컬렉션 name에 저장"""
    return Milvus.from_documents(
        chunks, embeddings, collection_name=name,
        connection_args={"uri": DB_URI},                               # 저장할 DB 파일
        index_params={"index_type": "FLAT", "metric_type": "COSINE"},  # 전수 비교 + 코사인 유사도
        ids=[c.metadata["chunk_id"] for c in chunks],                  # 청크 고유 번호를 ID로 사용
        drop_old=True)                                                 # 같은 이름이 있으면 지우고 새로 만듦

store = build_store(chunks, "mju_rules_500")          # GPU 기준 수 분 소요
print("저장 완료:", len(chunks), "건")

저장 완료: 4150 건


---
## [2단계] ⑤ 질문 인코딩 → ⑥ 유사 문서 검색 → ⑦ 응답 생성

- ⑤ 질문도 **같은 임베딩 모델**로 벡터화
- ⑥ 벡터 DB에서 질문과 **가장 비슷한 청크 K개**(여기서는 4개)를 찾음 → 앞의 숫자가 유사도 점수
- 결과 보기: 2위에 정답 근거인 **학칙 제50조**(134학점)가 검색됨

In [15]:
q = "명지대학교 인공지능·소프트웨어융합대학 학생은 졸업하려면 몇 학점을 취득해야 하나요?"

qv = embeddings.embed_query(q)                                   # ⑤ 질문 → 벡터
print("질문 벡터 차원:", len(qv))

for doc, score in store.similarity_search_with_score(q, k=4):    # ⑥ 비슷한 청크 4개 + 유사도 점수
    print(round(score, 3), doc.metadata["reg_id"], doc.page_content[:100])

질문 벡터 차원: 1024
0.631 2-1-1 [명지대학교 학칙] 제74조(준용) … 전공) 87 인공지능· 소프트웨어 융합대학 융합소프트웨어학부 110 응용소프트웨어전공 데이터사이언스전공 인공지능전공 디지털콘텐츠디자인학과 소
0.622 2-1-1 [명지대학교 학칙] 제50조(학사과정의 졸업) ①소정의 교육과정을 이수하고, 128학점(스마트시스템공과 대학, 반도체žICT대학, 인공지능ž소프트웨어융합대학은 134학점, 건축대학
0.597 2-1-1 [명지대학교 학칙] 제74조(준용) … 청소년지도학전공 아동학전공 소계 (2개 학부, 2개 전공) 87 경영대학 경영학부 182 경영학전공 글로벌비즈니스학전공(외) 경영정보학과 소
0.577 2-1-1 [명지대학교 학칙] 제74조(준용) … 데이터테크놀로지 전공) 60 60 60 60 60 60 소계 (1개 학부, 1개 학과, 2개 전공) 100 100 100 100 100 10


### ⑦ 응답 생성 — LCEL 체인
- **프롬프트 규칙** 3가지: ① 컨텍스트(검색된 청크)에 있는 내용만 사용 ② 근거가 없으면 "찾을 수 없습니다" ③ 마지막에 출처 표기
- **LCEL 체인**: `|`(파이프)로 부품을 **차례대로 연결**하는 LangChain 문법
  - `retriever`(청크 찾기) → `format_docs`(청크를 글로 묶기) → `RAG_PROMPT`(프롬프트 완성) → `llm`(답변) → `StrOutputParser`(글자만 꺼내기)
- 결과 보기: 이번에는 **134학점**으로 정확히 답하고 출처(학칙 제50조)까지 표기 → RAG의 효과

In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 프롬프트: system(규칙) + human(컨텍스트와 질문)
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "너는 명지대학교 규정 안내 도우미다. 반드시 [컨텍스트]에 있는 내용만 근거로 한국어로 간결하게 답하라.\n"
               "컨텍스트에 근거가 없으면 추측하지 말고 '규정에서 찾을 수 없습니다'라고 답하라.\n"
               "답변 마지막 줄에 (출처: 규정명 제N조) 형식으로 근거를 적어라."),
    ("human", "[컨텍스트]\n{context}\n\n[질문]\n{question}"),
])

def format_docs(docs):
    """검색된 청크들을 <1> … <2> … 형태의 글 하나로 묶기"""
    return "\n\n".join(f"<{i+1}> {d.page_content}" for i, d in enumerate(docs))

retriever = store.as_retriever(search_kwargs={"k": 4})          # 검색기: 질문 → 비슷한 청크 4개
rag_chain = ({"context": retriever | format_docs,               # 질문으로 청크를 찾아 글로 묶어 context에 넣고
              "question": RunnablePassthrough()}                # 질문은 그대로 question에 넣음
             | RAG_PROMPT | llm | StrOutputParser())             # ⑦ 프롬프트 → LLM → 답변 글자

print(rag_chain.invoke(q))

명지대학교 인공지능·소프트웨어융합대학 학생은 졸업하려면 **134학점**을 취득해야 합니다.

(출처: 명지대학교 학칙 제50조 제1항)


### 평가용 RAG 함수 `rag()`
- 위의 체인은 **답변만** 돌려줌 → 채점하려면 **검색된 청크**와 **걸린 시간**도 필요
- 그래서 결과를 사전(dict)으로 돌려주는 함수를 만듦: `answer`(답변), `docs`(검색된 청크), `search_ms`(검색 시간), `total_ms`(전체 시간)
- `search_fn`: 검색 방법을 **갈아 끼울 수 있게** 함수로 받음 (나중에 다른 검색 방식과 비교할 때 사용)
- `dense_search(q, k)`는 `retriever.invoke(q)`와 같은 일을 함 (K를 바꿔 가며 실험하기 쉽게 함수로 만든 것)

In [17]:
import time

def dense_search(q, k=4, store=store):
    """벡터 DB에서 질문과 비슷한 청크 k개 찾기 (= retriever.invoke와 같은 동작)"""
    return store.similarity_search(q, k=k)

def rag(question, search_fn=dense_search, k=4, prompt=RAG_PROMPT):
    t0 = time.time()                                   # 시작 시각
    docs = search_fn(question, k=k)                    # ⑥ 검색
    t1 = time.time()                                   # 검색 끝난 시각
    answer = (prompt | llm | StrOutputParser()).invoke({"context": format_docs(docs), "question": question})   # ⑦ 답변
    return {"answer": answer, "docs": docs,
            "search_ms": (t1 - t0) * 1000,             # 검색에 걸린 시간(밀리초)
            "total_ms": (time.time() - t0) * 1000}     # 전체 걸린 시간(밀리초)

out = rag("학부생은 도서관에서 책을 몇 권, 며칠 동안 빌릴 수 있나요?")
print(out["answer"])
print([d.metadata["title"] for d in out["docs"]], f"{out['total_ms']:.0f}ms")

학부과정 학생은 **20책, 14일** 동안 빌릴 수 있습니다.

(출처: 도서관 규정 제10조 제1항 제6호)
['도서관 규정', '도서관 규정', '도서관 규정', '도서관 규정'] 3404ms


---
## 기본(Naive) RAG의 3가지 한계 직접 보기

| 한계 | 무슨 뜻? | 아래 실험 |
|---|---|---|
| ① 질의 한계 | 단어가 비슷하면 다 가져옴 → 질문 의도(학부? 대학원?)가 모호하면 엉뚱한 규정이 섞임 | '휴학' 질문 → 학부·특수대학원·미래교육원 규정이 섞임 |
| ② 검색 한계 | 찾은 것을 걸러내지 않고 모두 LLM에 전달 → K가 클수록 노이즈·글자 수 증가 | K=2, 5, 10일 때 규정 종류·글자 수 비교 |
| ③ 선형 프로세스 한계 | 질문 → 검색 → 생성을 **한 번만** 함 → 여러 근거를 조합해야 하는 질문에 약함 | 졸업학점 ÷ 학기당 최대 학점 계산 질문 |

In [18]:
# ① 질의 한계: '휴학'은 학칙·시행규칙(학사/대학원/특수대학원)·외국인학생 규정 등 여러 곳에 등장
for d in dense_search("휴학은 최대 몇 학기까지 할 수 있나요?", k=5):
    print(d.metadata["reg_id"], d.metadata["title"], "|", d.page_content[:60].replace("\n", " "))

2-2-1 학칙 시행규칙(학사과정) | [학칙 시행규칙(학사과정)] 제16조(휴학) ①재학 중 질병, 의무군복무, 육아, 창업, 기타 부득이한 사유
2-1-1 명지대학교 학칙 | [명지대학교 학칙] 제19조(휴학) ①휴학하고자 하는 자는 소정기간 내에 휴학원을 제출하여 총장의 허 가를 
5-2-1 미래교육원 규정 | [미래교육원 규정] 제16조(휴학의 종류) 휴학의 종류는 다음과 같다. 1.일반휴학 : 질병, 사고, 가정사
2-2-1 학칙 시행규칙(학사과정) | [학칙 시행규칙(학사과정)] 제18조(일반휴학) ①일반휴학을 하고자 하는 자는 매 학기 소정기간 내에 휴학절
2-2-4 학칙 시행규칙(특수대학원) | [학칙 시행규칙(특수대학원)] 제17조(휴학의 종류) 휴학의 종류는 다음과 같다. 1.일반휴학 : 질병, 사


In [19]:
# ② 검색 한계: K가 커지면 서로 다른 규정이 섞이고 LLM에 전달되는 글자 수가 증가
q = "학생 징계의 종류에는 무엇이 있나요?"
for k in [2, 5, 10]:
    found = dense_search(q, k=k)
    print(f"K={k:2d} | 규정 종류 {len({d.metadata['reg_id'] for d in found})}개 | 컨텍스트 {len(format_docs(found)):5d}자")

K= 2 | 규정 종류 2개 | 컨텍스트   768자
K= 5 | 규정 종류 2개 | 컨텍스트  2069자
K=10 | 규정 종류 4개 | 컨텍스트  3905자


In [20]:
# ③ 선형 프로세스 한계: 졸업학점(제50조)과 학기당 최대 학점(제31조)을 모두 찾아 계산해야 하는 질문
q = "인공지능·소프트웨어융합대학 학생이 매 학기 최대 학점만 수강하면 졸업까지 최소 몇 학기가 필요한가요?"
out = rag(q)
print([d.page_content[:40] for d in out["docs"]])
print(out["answer"])

['[학·석사연계과정 운영에 관한 내규] 제6조(수업연한) 연계과정의 수업연', '[학칙 시행규칙(학사과정)] 제108조(졸업학점) ①학사과정의 졸업에 필', '[학사처리(학사과정)에 관한 내규] 제62조(적용) … 교환/방문학생프로', '[융합전공 운영에 관한 내규] 제9조(학점이수) ①융합전공 이수자는 졸업']
인공지능·소프트웨어융합대학 학생의 경우, [학·석사연계과정 운영에 관한 내규] 제6조에 따라 연계과정의 수업연한은 5년(60개월)이며, 학사학위과정은 3.5년(7개 학기)입니다. 따라서 매 학기 최대 학점만 수강하더라도 졸업까지 최소 **7개 학기**가 필요합니다.

다만, 이는 학·석사연계과정 내규에 따른 학사학위과정 수업연한 기준이며, [학칙 시행규칙(학사과정)] 제108조의 졸업학점 요건(인공지능·소프트웨어융합대학은 134학점 이상)과는 별도로 확인해야 합니다.

(출처: 학·석사연계과정 운영에 관한 내규 제6조)


> **결과 보기**: ③에서 필요한 근거(졸업학점 제50조 + 학기당 최대 학점 제31조)를 함께 찾지 못해 엉뚱한 근거(학·석사연계과정)로 답함 → 이런 문제를 Advanced·Modular RAG에서 해결

---
# Part B. RAG 성능 평가 (채점하기)

### 채점은 두 가지로 나눠서 함
| 구분 | 질문 | 지표 |
|---|---|---|
| **검색 평가** | 정답 근거가 들어 있는 청크를 **찾아왔나?** | Hit@K, Precision@K, Recall@K, MRR, NDCG@K, Context Precision |
| **응답 평가** | 찾은 자료를 바탕으로 **맞게 답했나?** | Faithfulness, Answer Relevancy, Correctness |

- 왜 나누나? 답이 틀렸을 때 **검색이 문제인지(못 찾음) 생성이 문제인지(찾았는데 잘못 답함)** 구분하려고

## 채점용 문제지(Golden Set) 만들기
- 문항 하나 = `question`(질문) + `ground_truth`(모범답안) + `evidence`(정답 근거: 규정번호, 근거 문구)
- 문항 유형: **사실형** 20개(근거 1개) / **종합형** 2개(근거 2개) / **정답없음** 3개(규정에 없는 질문)
- 근거 문구는 규정 원문에 **그대로 있는 짧은 문구** → 이 문구가 들어 있는 청크를 '정답 청크'로 판정

In [21]:
golden = [
    # ── 단일 문서 사실형 ──
    {"id": "q01", "type": "사실형", "question": "학사과정의 수업연한은 몇 년인가요?",
     "ground_truth": "4년이다. 다만 건축대학은 5년이다.", "evidence": [("2-1-1", "학사과정의 수업연한은 4년으로 한다")]},
    {"id": "q02", "type": "사실형", "question": "인공지능·소프트웨어융합대학 학생의 졸업 이수학점은?",
     "ground_truth": "134학점 이상 취득해야 졸업이 인정된다.", "evidence": [("2-1-1", "소프트웨어융합대학은 134학점")]},
    {"id": "q03", "type": "사실형", "question": "조기졸업을 하려면 평균평점이 얼마 이상이어야 하나요?",
     "ground_truth": "전체 성적 평균평점 4.0 이상(학·석사 연계과정은 3.5 이상)이고 졸업학점을 취득하면 한 학기 조기졸업할 수 있다.",
     "evidence": [("2-1-1", "한 학기 조기졸업을 할 수 있다")]},
    {"id": "q04", "type": "사실형", "question": "학사경고를 받는 성적 기준은 무엇인가요?",
     "ground_truth": "당해학기 평균평점 1.50 미만이면 학사경고를 받는다. 수강신청을 하지 않은 재학생도 학사경고를 받는다.",
     "evidence": [("2-1-1", "평균평점 1.50미만인 자에게는 학사경고")]},
    {"id": "q05", "type": "사실형", "question": "경영대학 학생은 한 학기에 최대 몇 학점까지 취득할 수 있나요?",
     "ground_truth": "17학점이다(계절수업, 채플 등은 산입하지 않음).", "evidence": [("2-1-1", "아너칼리지(전공자유대학) : 17학점")]},
    {"id": "q06", "type": "사실형", "question": "계절수업으로 취득할 수 있는 학점 한도는?",
     "ground_truth": "7학점 이내이다.", "evidence": [("2-1-1", "계절수업 취득학점은 7학점 이내로 한다")]},
    {"id": "q07", "type": "사실형", "question": "원격수업은 개설 교과목 학점의 몇 퍼센트까지 운영할 수 있나요?",
     "ground_truth": "총 개설 교과목 학점수의 100분의 20(20%)을 초과할 수 없다.", "evidence": [("2-1-1", "100분의 20을 초과할 수 없다")]},
    {"id": "q08", "type": "사실형", "question": "학부생은 재학 중 일반휴학을 통산 몇 학기까지 할 수 있나요?",
     "ground_truth": "6개 학기를 초과할 수 없다(의무군복무·육아·창업 휴학은 제외).", "evidence": [("2-2-1", "일반휴학은 6개 학기를 초과할 수 없다")]},
    {"id": "q09", "type": "사실형", "question": "재수강은 어떤 성적의 과목에 가능하고 최고 몇 점까지 받을 수 있나요?",
     "ground_truth": "C+ 이하 과목만 학기당 최대 2과목, 과목당 최대 3회까지 가능하며 최고 성적은 A0이다.",
     "evidence": [("2-2-1", "재수강시 취득가능 최고 성적은 A0")]},
    {"id": "q10", "type": "사실형", "question": "상대평가 시 A학점 비율은 어떻게 되나요?",
     "ground_truth": "A학점 10~40%, B학점 20~40%, C~F학점 20~70%를 원칙으로 하며 A·B학점을 80%까지 상향할 수 있다.",
     "evidence": [("2-2-1", "A학점 10-40%")]},
    {"id": "q11", "type": "사실형", "question": "학생 징계의 종류에는 무엇이 있나요?",
     "ground_truth": "경고, 근신, 유기정학, 무기정학, 제적으로 구분한다.", "evidence": [("3-2-3", "경고, 근신, 유기정학, 무기정학, 제적으로 구분한다")]},
    {"id": "q12", "type": "사실형", "question": "학부생은 도서관에서 책을 몇 권, 며칠 동안 빌릴 수 있나요?",
     "ground_truth": "20책, 14일이다.", "evidence": [("5-1-1", "학부과정 학생 : 20책 14일")]},
    {"id": "q13", "type": "사실형", "question": "도서 반납이 연체되면 어떤 제재를 받나요?",
     "ground_truth": "반납 시까지 대출이 중지되고, 연체일수의 2배 기간 동안 대출이 중지된다. 180일을 초과하면 분실로 간주해 변상한다.",
     "evidence": [("5-1-1", "연체일수의 2배일에 해당하는 기간동안 대출을 중지")]},
    {"id": "q14", "type": "사실형", "question": "명예졸업은 어떤 사람에게 수여하나요?",
     "ground_truth": "재학 당시 품행이 방정했으나 특별한 사정이나 불의의 사고로 학업을 마치지 못한 사람 중 사회·국가 또는 대학 발전에 공헌했거나 명예졸업심사위원회 심의를 통과한 사람이다.",
     "evidence": [("3-1-36", "불의의 사고로 인하여 학업을 마치지 못한 사람")]},
    {"id": "q15", "type": "사실형", "question": "학생단체를 등록하려면 정규회원이 몇 명 이상 있어야 하나요?",
     "ground_truth": "20명 이상의 정규회원이 필요하다.", "evidence": [("3-2-2", "20명 이상의 입회원서")]},
    {"id": "q16", "type": "사실형", "question": "학생이 교내 집회를 열려면 언제까지 허가를 신청해야 하나요?",
     "ground_truth": "집회 및 행사 7일 전까지 학생처장에게 허가 신청서를 제출해 허가를 받아야 한다.",
     "evidence": [("3-2-1", "집회 및 행사예정 7일전까지")]},
    {"id": "q17", "type": "사실형", "question": "교수 연구년은 신규 임용 후 몇 년을 근무해야 신청할 수 있나요?",
     "ground_truth": "신규 임용 후 6년 이상 근무하면 6월 또는 1년의 연구년을 신청할 수 있다.",
     "evidence": [("4-2-11", "신규 임용 후 6년 이상 근무한 교원")]},
    {"id": "q18", "type": "사실형", "question": "연구년을 다녀온 교원은 복귀 후 얼마 동안 근무해야 하나요?",
     "ground_truth": "연구년 기간의 3배수에 해당하는 기간 동안 근무해야 한다.", "evidence": [("4-2-11", "연구년 기간의 3배수")]},
    {"id": "q19", "type": "사실형", "question": "교내에서 전동킥보드 음주운전으로 보는 혈중알코올농도 기준은?",
     "ground_truth": "혈중알코올농도 0.03% 이상이다.", "evidence": [("4-4-32", "혈중알코올농도가 0.03퍼센트 이상")]},
    {"id": "q20", "type": "사실형", "question": "개인 소유 전동킥보드를 교내 등록하면 유효기간은 얼마인가요?",
     "ground_truth": "등록일로부터 1년이며, 종료일이 학기 중이면 해당 학기 말까지이다.", "evidence": [("4-4-32", "운행 유효기간은 등록일로부터 1년")]},
    # ── 복수 근거 종합형 ──
    {"id": "q21", "type": "종합형", "question": "학부생은 한 번에 몇 학기까지 휴학할 수 있고, 재학 중 일반휴학은 통산 몇 학기까지 가능한가요?",
     "ground_truth": "1회에 두 학기를 초과할 수 없고, 재학 중 일반휴학은 통산 6개 학기를 초과할 수 없다.",
     "evidence": [("2-1-1", "휴학은 1회에 두 학기를 초과하지 못하며"), ("2-2-1", "일반휴학은 6개 학기를 초과할 수 없다")]},
    {"id": "q22", "type": "종합형", "question": "성적을 인정받으려면 수업에 얼마나 출석해야 하고, 군복무 휴학생은 기준이 어떻게 다른가요?",
     "ground_truth": "수업일수의 5분의 4 이상 출석해야 성적이 인정된다. 의무군복무·육아·창업 휴학의 경우 3분의 2 기준이 적용된다.",
     "evidence": [("2-1-1", "수업일수 5분의 4 이상을 출석한 자에 한하여"), ("2-2-1", "3분의 2미만일 경우 성적을 인정하지 아니한다")]},
    # ── 정답 없는 질문 ──
    {"id": "n01", "type": "정답없음", "question": "2027학년도 명지대학교 등록금 인상률은 몇 퍼센트인가요?",
     "ground_truth": "규정에 해당 정보가 없으므로 알 수 없다고 답해야 한다.", "evidence": []},
    {"id": "n02", "type": "정답없음", "question": "이번 주 학생식당 점심 메뉴는 무엇인가요?",
     "ground_truth": "규정에 해당 정보가 없으므로 알 수 없다고 답해야 한다.", "evidence": []},
    {"id": "n03", "type": "정답없음", "question": "기숙사 입사 신청 시 통금 시간은 몇 시인가요?",
     "ground_truth": "규정에 해당 정보가 없으므로 알 수 없다고 답해야 한다.", "evidence": []},
]
pd.DataFrame(golden)["type"].value_counts()

,count
type,
사실형,20
정답없음,3
종합형,2


### 근거 문구가 정말 규정에 있는지 확인
- 문제지의 근거가 잘못되면 채점도 엉터리 → 먼저 확인 (모두 `O`면 정상)
- 비교할 때 **공백을 모두 지움** → 줄바꿈이나 띄어쓰기 차이로 못 찾는 일을 막음

In [22]:
def norm(text):
    return re.sub(r"\s+", "", text)            # 공백·줄바꿈 모두 제거

doc_by_id = {d.metadata["reg_id"]: norm(d.page_content) for d in docs}   # 규정번호 → 공백 없앤 원문
for item in golden:
    for src, ev in item["evidence"]:
        print(item["id"], src, "O" if norm(ev) in doc_by_id.get(src, "") else "X  ← 근거 문구 확인 필요", ev)

q01 2-1-1 O 학사과정의 수업연한은 4년으로 한다
q02 2-1-1 O 소프트웨어융합대학은 134학점
q03 2-1-1 O 한 학기 조기졸업을 할 수 있다
q04 2-1-1 O 평균평점 1.50미만인 자에게는 학사경고
q05 2-1-1 O 아너칼리지(전공자유대학) : 17학점
q06 2-1-1 O 계절수업 취득학점은 7학점 이내로 한다
q07 2-1-1 O 100분의 20을 초과할 수 없다
q08 2-2-1 O 일반휴학은 6개 학기를 초과할 수 없다
q09 2-2-1 O 재수강시 취득가능 최고 성적은 A0
q10 2-2-1 O A학점 10-40%
q11 3-2-3 O 경고, 근신, 유기정학, 무기정학, 제적으로 구분한다
q12 5-1-1 O 학부과정 학생 : 20책 14일
q13 5-1-1 O 연체일수의 2배일에 해당하는 기간동안 대출을 중지
q14 3-1-36 O 불의의 사고로 인하여 학업을 마치지 못한 사람
q15 3-2-2 O 20명 이상의 입회원서
q16 3-2-1 O 집회 및 행사예정 7일전까지
q17 4-2-11 O 신규 임용 후 6년 이상 근무한 교원
q18 4-2-11 O 연구년 기간의 3배수
q19 4-4-32 O 혈중알코올농도가 0.03퍼센트 이상
q20 4-4-32 O 운행 유효기간은 등록일로부터 1년
q21 2-1-1 O 휴학은 1회에 두 학기를 초과하지 못하며
q21 2-2-1 O 일반휴학은 6개 학기를 초과할 수 없다
q22 2-1-1 O 수업일수 5분의 4 이상을 출석한 자에 한하여
q22 2-2-1 O 3분의 2미만일 경우 성적을 인정하지 아니한다


---
## 검색 평가 지표 (Retrieval Metrics)

| 지표 | 정의 | 쉬운 말로 |
|---|---|---|
| Hit@K | 상위 K개에 관련 청크가 1개 이상이면 1 | 정답을 **하나라도** 찾았나? |
| Precision@K | 상위 K개 중 관련 청크 비율 (노이즈 진단) | 가져온 것 중 **쓸모 있는 것**의 비율 |
| Recall@K | 전체 정답 근거 중 상위 K개에 회수된 비율 (누락 진단) | 필요한 정답을 **빠짐없이** 가져왔나? |
| MRR | 첫 관련 청크 순위의 역수 평균 | 정답이 **얼마나 위에** 있나? (1위 1.0, 2위 0.5, 3위 0.33) |
| NDCG@K | 순위로 할인(log₂)한 DCG ÷ 이상적 순서의 IDCG | 정답들이 **이상적인 순서**에 얼마나 가깝나? |
| Context Precision | 관련 청크가 나온 위치마다의 Precision을 평균 (RAGAS 방식, 순위 가중 정밀도) | 정답이 **앞쪽에 몰려 있을수록** 높음 |

### LangChain에는 이 지표 함수가 없음 → 직접 만들기
- LangChain은 검색(`retriever.invoke`)까지만 해 주고, 위 6개 지표를 계산하는 **내장 함수는 없음**
- 그래서 아래처럼 **지표 하나 = 함수 하나**로 직접 만듦
- 모든 함수의 입력은 같음
  - `retrieved`: 검색 결과를 **순위대로** 적은 목록 (예: `['c1', 'c7', 'c3', 'c9']`)
  - `relevant`: **정답 청크** 모음 (예: `{'c1', 'c3', 'c5'}`)
  - `k`: 상위 몇 개까지 볼지
- 같은 근거가 두 청크에 겹쳐 들어 있을 수 있음 → Recall·NDCG는 **같은 근거를 처음 가져온 한 번만** 점수로 인정

In [ ]:
import math

def hit_at_k(retrieved, relevant, k):
    """Hit@K: 상위 K개 안에 정답 청크가 하나라도 있으면 1, 없으면 0"""
    top_k = retrieved[:k]                       # 앞에서부터 K개만 봄
    for chunk in top_k:
        if chunk in relevant:                   # 정답 청크를 하나라도 만나면
            return 1
    return 0

def precision_at_k(retrieved, relevant, k):
    """Precision@K: 상위 K개 중 정답 청크의 비율 (낮으면 노이즈가 많음)"""
    top_k = retrieved[:k]
    n_relevant = sum(1 for chunk in top_k if chunk in relevant)   # 정답 청크 개수 세기
    return n_relevant / k

def recall_at_k(retrieved, relevant, k):
    """Recall@K: 필요한 정답 중 상위 K개 안에 가져온 비율 (낮으면 누락이 있음)"""
    if len(relevant) == 0:
        return 0.0
    found = set(retrieved[:k]) & set(relevant)  # 가져온 정답 (겹치면 한 번만 셈)
    return len(found) / len(relevant)

def reciprocal_rank(retrieved, relevant, k):
    """RR: 첫 번째 정답 청크 순위의 역수 (1위→1.0, 2위→0.5, 3위→0.33). 문항 전체 평균 = MRR"""
    for rank, chunk in enumerate(retrieved[:k], start=1):   # rank: 1, 2, 3, ...
        if chunk in relevant:
            return 1 / rank
    return 0.0

def ndcg_at_k(retrieved, relevant, k):
    """NDCG@K: 실제 순서의 점수(DCG) ÷ 가장 이상적인 순서의 점수(IDCG)"""
    dcg, seen = 0.0, set()
    for rank, chunk in enumerate(retrieved[:k], start=1):
        if chunk in relevant and chunk not in seen:   # 같은 정답은 처음 한 번만 인정
            dcg += 1 / math.log2(rank + 1)            # 아래 순위일수록 점수를 깎음 (1위 1, 2위 0.63, 3위 0.5 …)
            seen.add(chunk)
    n_ideal = min(len(relevant), k)                   # 이상적: 정답이 1위부터 꽉 채워진 경우
    idcg = sum(1 / math.log2(rank + 1) for rank in range(1, n_ideal + 1))
    return dcg / idcg if idcg > 0 else 0.0

def context_precision_at_k(retrieved, relevant, k):
    """Context Precision@K (RAGAS 방식): 정답 청크가 나온 위치마다 Precision을 구해 평균"""
    hits, precision_sum = 0, 0.0
    for rank, chunk in enumerate(retrieved[:k], start=1):
        if chunk in relevant:
            hits += 1
            precision_sum += hits / rank              # 이 위치까지의 Precision
    return precision_sum / hits if hits > 0 else 0.0

def retrieval_metrics(retrieved, relevant, k):
    """6개 지표를 한 번에 계산해 사전(dict)으로 돌려줌"""
    return {
        "hit": hit_at_k(retrieved, relevant, k),
        "precision": precision_at_k(retrieved, relevant, k),
        "recall": recall_at_k(retrieved, relevant, k),
        "mrr": reciprocal_rank(retrieved, relevant, k),
        "ndcg": ndcg_at_k(retrieved, relevant, k),
        "ctx_precision": context_precision_at_k(retrieved, relevant, k),
    }

### 강의자료 계산 예시로 검증하기
- 조건: K=4, 정답 청크 3개(`c1`, `c3`, `c5`), 검색 결과 1위 `c1`(정답) 2위 `c7`(무관) 3위 `c3`(정답) 4위 `c9`(무관)
- 손으로 계산하면
  - Hit@4 = 1 / Precision@4 = 2÷4 = 0.5 / Recall@4 = 2÷3 = 0.67 / MRR = 1÷1 = 1.0
  - NDCG@4 = (1 + 1/log₂4) ÷ (1 + 1/log₂3 + 1/log₂4) = 1.5 ÷ 2.13 = **0.70** → 강의자료 종합 레퍼런스의 'NDCG 70%'
  - Context Precision@4 = (1/1 + 2/3) ÷ 2 = **0.83**
- 참고: IDCG를 '검색된 정답 2개'로만 계산하면 0.92 (표준 정의는 전체 정답 3개 기준 → 0.70)

In [ ]:
retrieved = ["c1", "c7", "c3", "c9"]      # 검색 결과 (순위대로)
relevant = {"c1", "c3", "c5"}             # 정답 청크

for name, f in [("Hit@4", hit_at_k), ("Precision@4", precision_at_k), ("Recall@4", recall_at_k),
                ("MRR", reciprocal_rank), ("NDCG@4", ndcg_at_k), ("Context Precision@4", context_precision_at_k)]:
    print(f"{name:20s} {f(retrieved, relevant, k=4):.2f}")

print("\nIDCG를 검색된 정답 2개로 계산:", round((1 + 1 / math.log2(4)) / (1 + 1 / math.log2(3)), 2))

Hit@4                1.00
Precision@4          0.50
Recall@4             0.67
MRR                  1.00
NDCG@4               0.70
Context Precision@4  0.83

IDCG를 검색된 정답 2개로 계산: 0.92


### 실제 RAG에 적용하기 — 검색된 청크에 '근거 번호' 붙이기
- 문제지에는 청크 ID가 아니라 **근거 문구**가 있음 → 검색된 청크마다 **어떤 근거가 들어 있는지** 확인해 이름표를 붙임
  - 근거 문구가 들어 있으면 `'근거1'`, `'근거2'` … / 없으면 `'무관'`
  - 정답 모음 `relevant` = `{'근거1', '근거2', …}` (문항의 근거 개수만큼)
- 그러면 위에서 만든 지표 함수를 **그대로** 사용 가능
- `eval_retrieval`: 문제지의 모든 문항을 검색해서 문항별 점수표(DataFrame)를 만듦 (정답없음 문항은 제외)

In [25]:
def label_docs(docs, item):
    """검색된 청크마다 '근거N' 또는 '무관' 이름표를 붙인 목록을 돌려줌"""
    labels = []
    for d in docs:
        label = "무관"
        for n, (src, ev) in enumerate(item["evidence"], start=1):
            # 같은 규정이고, 근거 문구가 청크 안에 들어 있으면 → 근거N
            if d.metadata["reg_id"] == src and norm(ev) in norm(d.page_content):
                label = f"근거{n}"
                break
        labels.append(label)
    return labels

def relevant_set(item):
    """문항의 정답 모음: {'근거1', '근거2', ...}"""
    return {f"근거{n}" for n in range(1, len(item["evidence"]) + 1)}

def eval_retrieval(search_fn, k=4, items=golden):
    rows = []
    for item in items:
        if not item["evidence"]:                       # 정답 없는 질문은 검색 평가에서 제외
            continue
        t0 = time.time()
        found = search_fn(item["question"], k=k)      # 검색
        ms = (time.time() - t0) * 1000
        labels = label_docs(found, item)               # 예: ['근거1', '무관', '무관', '무관']
        rows.append({"id": item["id"], "type": item["type"],
                     **retrieval_metrics(labels, relevant_set(item), k),   # 6개 지표
                     "ctx_chars": len(format_docs(found)),                 # LLM에 넘길 글자 수
                     "search_ms": ms})                                     # 검색 시간
    return pd.DataFrame(rows)

r = eval_retrieval(dense_search, k=4)
r.round(2)

,id,type,hit,precision,recall,mrr,ndcg,ctx_precision,ctx_chars,search_ms
0,q01,사실형,1,0.25,1.0,1.00,1.00,1.00,1661,2049.39
1,q02,사실형,0,0.00,0.0,0.00,0.00,0.00,1904,2124.97
2,q03,사실형,1,0.25,1.0,0.33,0.50,0.33,1244,2156.69
3,q04,사실형,1,0.25,1.0,1.00,1.00,1.00,1691,2097.39
4,q05,사실형,1,0.25,1.0,0.33,0.50,0.33,1967,2153.19
5,q06,사실형,1,0.25,1.0,0.50,0.63,0.50,1749,2182.66
6,q07,사실형,0,0.00,0.0,0.00,0.00,0.00,1320,2060.35
7,q08,사실형,1,0.25,1.0,0.33,0.50,0.33,1860,2224.49
8,q09,사실형,1,0.25,1.0,1.00,1.00,1.00,1599,2090.28
9,q10,사실형,1,0.25,1.0,1.00,1.00,1.00,1654,2070.86


> **결과 보기**: 대부분 Hit=1(정답을 찾음)이지만 q02·q07은 Hit=0(정답 청크를 상위 4개 안에 못 가져옴). MRR이 0.33인 문항은 정답이 3위에 있었다는 뜻

### 참고: LangChain retriever로 한 문항 직접 계산해 보기
- 실무에서 흔히 쓰는 모양: `retriever.invoke(질문)` → 청크 목록 → 이름표 → 지표 함수

In [ ]:
item = golden[1]                                   # q02: 인공지능·소프트웨어융합대학 졸업 이수학점
found = retriever.invoke(item["question"])         # LangChain 검색기 (K=4)
labels = label_docs(found, item)

print("질문:", item["question"])
for rank, (d, label) in enumerate(zip(found, labels), start=1):
    print(f"{rank}위 [{label}] {d.metadata['chunk_id']} {d.page_content[:50]}")
print({m: round(v, 2) for m, v in retrieval_metrics(labels, relevant_set(item), k=4).items()})

### 평균 검색 지표 (Dense 검색, K=4)
- `METRICS`: 앞으로 계속 쓸 지표 이름 목록

In [26]:
METRICS = ["hit", "precision", "recall", "mrr", "ndcg", "ctx_precision"]

r[METRICS].mean().round(3)

,0
hit,0.909
precision,0.250
recall,0.909
mrr,0.761
ndcg,0.793
ctx_precision,0.750


---
## 응답 평가 지표 (Response Metrics) — LLM이 채점 (LLM-as-Judge)

| 지표 | 질문 | 필요한 것 | 계산 방법 (쉽게) |
|---|---|---|---|
| Faithfulness (충실성) | 찾은 자료에 **있는 말만** 했나? (환각 확인) | 답변, 청크 | 채점 LLM이 답변을 문장(주장)으로 나누고, 자료로 확인되는 주장의 비율 |
| Answer Relevancy (응답 관련성) | **묻는 말에** 답했나? | 질문, 답변 | 답변을 보고 질문 3개를 거꾸로 만들어서, 원래 질문과 얼마나 비슷한지 평균 |
| Correctness (정확성) | **모범답안과 같나?** | 답변, 모범답안 | 채점 LLM이 핵심 사실(숫자·기간 등)이 맞는지 True/False |
| Negative Rejection | 모르는 건 **모른다고** 했나? | 답변 | '정답없음' 문항에서의 Correctness |

- `with_structured_output(…)`: 채점 결과를 **정해진 모양(Pydantic 클래스)** 으로 받음 → 글자를 해석하다 생기는 오류를 막음

In [27]:
import numpy as np
from pydantic import BaseModel, Field

# 채점 LLM에게 받을 결과의 '모양' 정의
class Claims(BaseModel):                 # Faithfulness용: 주장 목록 + 각 주장이 근거 있는지
    claims: list[str] = Field(description="답변을 구성하는 개별 사실 주장 목록 (출처 표기 문장은 제외)")
    supported: list[bool] = Field(description="각 주장이 컨텍스트로 뒷받침되면 true")

class GenQuestions(BaseModel):           # Answer Relevancy용: 거꾸로 만든 질문 3개
    questions: list[str] = Field(description="답변을 보고 역으로 추정한 질문 3개")

class Verdict(BaseModel):                # Correctness용: 맞음/틀림 + 이유
    correct: bool = Field(description="응답이 정답과 사실적으로 일치하면 true")
    reason: str = Field(description="판정 이유 한 문장")

def faithfulness(answer, docs):
    """근거 있는 주장 수 ÷ 전체 주장 수 (1.0이면 환각 없음)"""
    r = judge.with_structured_output(Claims, method="function_calling").invoke(
        f"[컨텍스트]\n{format_docs(docs)}\n\n[답변]\n{answer}\n\n"
        "답변을 개별 사실 주장으로 나누고, 각 주장이 컨텍스트로 뒷받침되는지 판정하라. "
        "'찾을 수 없다'처럼 정보가 없다는 답변은 주장이 없는 것으로 본다.")
    return float(np.mean(r.supported)) if r.supported else 1.0     # 주장이 없으면 1.0

def answer_relevancy(question, answer):
    """답변으로 거꾸로 만든 질문 3개와 원래 질문의 평균 유사도"""
    r = judge.with_structured_output(GenQuestions, method="function_calling").invoke(
        f"[답변]\n{answer}\n\n이 답변이 대답하고 있는 질문을 한국어로 3개 만들어라.")
    qv = np.array(embeddings.embed_query(question))               # 원래 질문 벡터
    gv = np.array(embeddings.embed_documents(r.questions))        # 만든 질문 3개 벡터
    return float((gv @ qv).mean())                     # 길이 1인 벡터끼리 내적 = 코사인 유사도

def correctness(question, answer, ground_truth):
    """모범답안과 핵심 사실이 같으면 True"""
    r = judge.with_structured_output(Verdict, method="function_calling").invoke(
        f"[질문]\n{question}\n\n[정답]\n{ground_truth}\n\n[응답]\n{answer}\n\n"
        "응답이 정답의 핵심 사실(숫자, 기간, 대상 등)과 일치하는지 판정하라. 표현 차이나 추가 설명은 허용한다. "
        "정답이 '알 수 없다'인 경우 응답도 정보가 없다고 답해야 true이다.")
    return r.correct, r.reason

- 한 문항(q12 도서관 대출)으로 응답 지표 3개를 먼저 계산해 보기

In [28]:
item = golden[11]                                       # q12 도서관 대출
out = rag(item["question"])
print(out["answer"], "\n")
print("Faithfulness    :", faithfulness(out["answer"], out["docs"]))
print("Answer Relevancy:", round(answer_relevancy(item["question"], out["answer"]), 3))
print("Correctness     :", correctness(item["question"], out["answer"], item["ground_truth"]))

학부과정 학생은 **20책, 14일** 동안 빌릴 수 있습니다.

(출처: 도서관 규정 제10조 제1항 제6호) 

Faithfulness    : 1.0
Answer Relevancy: 0.793
Correctness     : (True, "응답이 정답의 핵심 사실인 '20책, 14일'과 정확히 일치하며, 학부생 대상이라는 점도 동일하게 전달하고 있다.")


### 전체 채점 함수 `evaluate()`
- 문항마다: RAG 실행 → 검색 지표 6개 + 응답 지표 3개 + 시간·글자 수
- `format_ok`: 프롬프트에서 요구한 `(출처: …)` 표기를 지켰는지 → FP5(잘못된 포맷) 진단에 사용
- `summary()`: 전체 평균. `negative_rejection` = 정답없음 문항에서 '모른다'고 답한 비율
- 25문항 × (답변 1번 + 채점 3번) LLM 호출 → 수 분 소요

In [29]:
def evaluate(search_fn=dense_search, k=4, prompt=RAG_PROMPT, items=golden):
    rows = []
    for item in tqdm(items):
        out = rag(item["question"], search_fn, k, prompt)                    # RAG 실행
        row = {"id": item["id"], "type": item["type"], "answer": out["answer"]}
        if item["evidence"]:                                                  # 검색 지표 (정답없음 문항 제외)
            labels = label_docs(out["docs"], item)
            row.update(retrieval_metrics(labels, relevant_set(item), k))
        row["faithfulness"] = faithfulness(out["answer"], out["docs"])        # 응답 지표
        row["relevancy"] = answer_relevancy(item["question"], out["answer"])
        row["correct"], row["reason"] = correctness(item["question"], out["answer"], item["ground_truth"])
        row["format_ok"] = "출처" in out["answer"]                            # 출처 표기 여부
        row.update(ctx_chars=len(format_docs(out["docs"])), total_ms=out["total_ms"])
        rows.append(row)
    return pd.DataFrame(rows)

def summary(df):
    """전체 평균 점수"""
    s = df[METRICS + ["faithfulness", "relevancy", "correct", "format_ok", "ctx_chars", "total_ms"]].mean(numeric_only=True)
    s["negative_rejection"] = df.loc[df["type"] == "정답없음", "correct"].mean()
    return s

result = evaluate(dense_search, k=4)
result[["id", "type", "hit", "precision", "recall", "mrr", "faithfulness", "relevancy", "correct"]].round(2)

  0%|          | 0/25 [00:00<?, ?it/s]

,id,type,hit,precision,recall,mrr,faithfulness,relevancy,correct
0,q01,사실형,1.0,0.25,1.0,1.00,1.0,0.72,True
1,q02,사실형,0.0,0.00,0.0,0.00,1.0,0.89,True
2,q03,사실형,1.0,0.25,1.0,0.33,1.0,0.81,True
3,q04,사실형,1.0,0.25,1.0,1.00,1.0,0.74,True
4,q05,사실형,1.0,0.25,1.0,0.33,1.0,0.86,True
5,q06,사실형,1.0,0.25,1.0,0.50,1.0,0.91,False
6,q07,사실형,0.0,0.00,0.0,0.00,1.0,0.68,False
7,q08,사실형,1.0,0.25,1.0,0.33,1.0,0.81,True
8,q09,사실형,1.0,0.25,1.0,1.00,1.0,0.77,True
9,q10,사실형,1.0,0.25,1.0,1.00,1.0,0.74,True


In [30]:
summary(result).round(3)

,0
hit,0.909
precision,0.250
recall,0.909
mrr,0.761
ndcg,0.793
ctx_precision,0.750
faithfulness,0.960
relevancy,0.731
correct,0.880
format_ok,0.960


> **결과 보기**: Hit 91% · Recall 91%로 검색은 양호하지만 Precision 25% → 4개 중 3개는 노이즈. Faithfulness 0.96(환각 거의 없음), Correctness 88%, 정답없음 질문은 100% '모른다'고 답함

---
## 사례 분석 ① 성공 사례 — 검색과 응답이 모두 성공
- 좋은 검색은 좋은 답변의 **필요조건** (좋은 검색이 있어야 좋은 답변이 가능. 하지만 검색이 좋다고 답변이 꼭 좋은 건 아님)
- 조건: Hit=1, MRR=1(정답이 1위), Faithfulness=1.0, Correctness=True인 문항 찾기

In [31]:
ok = result[(result["hit"] == 1) & (result["mrr"] == 1) & (result["faithfulness"] == 1.0) & (result["correct"])]
print("동시 성공 문항:", list(ok["id"]))

row = (ok if len(ok) else result).iloc[0]
item = next(g for g in golden if g["id"] == row["id"])
print("\n질문:", item["question"])
print("정답:", item["ground_truth"])
print("응답:", row["answer"])
print({m: round(row[m], 2) for m in ["hit", "precision", "recall", "mrr", "ndcg", "faithfulness"]}, "correct =", row["correct"])

동시 성공 문항: ['q01', 'q04', 'q09', 'q10', 'q12', 'q13', 'q14', 'q15', 'q16', 'q17', 'q18', 'q19', 'q20', 'q21', 'q22']

질문: 학사과정의 수업연한은 몇 년인가요?
정답: 4년이다. 다만 건축대학은 5년이다.
응답: 학사과정의 수업연한은 **4년**입니다.  
다만, 건축대학의 수업연한은 **5년**입니다.

(출처: 명지대학교 학칙 제11조 제1항)
{'hit': np.float64(1.0), 'precision': np.float64(0.25), 'recall': np.float64(1.0), 'mrr': np.float64(1.0), 'ndcg': np.float64(1.0), 'faithfulness': np.float64(1.0)} correct = True


---
## 사례 분석 ② 정답이 없는 질문
- 규정에 없는 질문(등록금 인상률, 식당 메뉴 등)에 LLM이 **억지로 답하면 환각**
- 이때는 **"모른다"고 답해야 정답** (Correctness = True)
- 해결책 두 가지
  1. **프롬프트 규칙**: "컨텍스트에 없으면 모른다고 답하라" → 아래에서 규칙 있음/없음 비교
  2. **유사도 임계값(Threshold)**: 가장 비슷한 청크의 점수가 기준보다 낮으면 LLM을 부르지 않고 바로 거부

In [32]:
NO_RULE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "너는 명지대학교 안내 도우미다. 아래 컨텍스트를 참고해서 질문에 친절하게 한국어로 답하라."),
    ("human", "[컨텍스트]\n{context}\n\n[질문]\n{question}"),
])

negatives = [g for g in golden if g["type"] == "정답없음"]
for name, prompt in [("규칙 없음", NO_RULE_PROMPT), ("규칙 있음", RAG_PROMPT)]:
    print(f"===== 프롬프트 {name} =====")
    for item in negatives:
        ans = rag(item["question"], dense_search, 4, prompt)["answer"]
        ok, _ = correctness(item["question"], ans, item["ground_truth"])
        print(f"[{'거부 성공' if ok else '환각 위험'}] {item['question']}\n    → {ans[:120]}")

===== 프롬프트 규칙 없음 =====
[거부 성공] 2027학년도 명지대학교 등록금 인상률은 몇 퍼센트인가요?
    → 2027학년도 명지대학교 등록금 인상률에 대한 정보는 제공된 컨텍스트에 포함되어 있지 않습니다. 컨텍스트에는 2026·2027학년도 입학정원(학부·학과·전공별 모집 인원)과 석사과정 입학정원만 나와 있으며, 등록금 
[거부 성공] 이번 주 학생식당 점심 메뉴는 무엇인가요?
    → 죄송하지만, 제공된 컨텍스트에는 학생식당 점심 메뉴에 대한 정보가 포함되어 있지 않습니다. 해당 내용은 학칙, 교양 과목, 학사 처리 규정 등과 관련된 자료라서 메뉴 정보를 확인할 수 없습니다.

학생식당 점심 메뉴
[거부 성공] 기숙사 입사 신청 시 통금 시간은 몇 시인가요?
    → 기숙사 입사 신청 시 통금 시간에 대한 정보는 제공된 컨텍스트에 포함되어 있지 않습니다.  
제공된 자료는 도서관 규정, 미래교육원 규정, 한국기록관리학교육원 규정, 명지대학교 학칙이며, 기숙사 통금 시간 관련 내용
===== 프롬프트 규칙 있음 =====
[거부 성공] 2027학년도 명지대학교 등록금 인상률은 몇 퍼센트인가요?
    → 규정에서 찾을 수 없습니다. (출처: 규정명 제74조)
[거부 성공] 이번 주 학생식당 점심 메뉴는 무엇인가요?
    → 규정에서 찾을 수 없습니다. (출처: 규정명 제N조)
[거부 성공] 기숙사 입사 신청 시 통금 시간은 몇 시인가요?
    → 규정에서 찾을 수 없습니다. (출처: 규정명 제N조)


### 유사도 임계값 정하기
- 정답 있는 질문과 정답 없는 질문의 **1위 청크 유사도**를 비교 → 정답없음 질문은 유사도가 낮은 편(평균 0.49 vs 0.66)

In [33]:
# 유사도 임계값: 정답 있는 질문과 없는 질문의 '최고 유사도' 분포 비교
top_scores = pd.DataFrame([
    {"id": g["id"], "type": g["type"], "top1": store.similarity_search_with_score(g["question"], k=1)[0][1]}
    for g in golden])
top_scores.groupby("type")["top1"].describe()[["min", "mean", "max"]].round(3)

,min,mean,max
type,,,
사실형,0.493,0.664,0.758
정답없음,0.420,0.493,0.540
종합형,0.650,0.654,0.657


In [34]:
# 정답 없는 질문의 최고 유사도보다 약간 높게 설정 (실무에서는 평가셋과 별도의 검증용 질문으로 정함)
THRESHOLD = round(top_scores.loc[top_scores["type"] == "정답없음", "top1"].max() + 0.01, 3)
print("THRESHOLD =", THRESHOLD)
print("임계값에 걸리는 정답 있는 질문:", list(top_scores.loc[(top_scores["type"] != "정답없음") & (top_scores["top1"] < THRESHOLD), "id"]))

def rag_with_threshold(question, k=4, th=THRESHOLD):
    hits = [(d, s) for d, s in store.similarity_search_with_score(question, k=k) if s >= th]
    if not hits:
        return "규정에서 찾을 수 없습니다. (유사도 임계값 미달)"
    return (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs([d for d, _ in hits]), "question": question})

for item in negatives + golden[:2]:
    print(item["id"], "→", rag_with_threshold(item["question"])[:80])

THRESHOLD = 0.55
임계값에 걸리는 정답 있는 질문: ['q19']
n01 → 규정에서 찾을 수 없습니다. (유사도 임계값 미달)
n02 → 규정에서 찾을 수 없습니다. (유사도 임계값 미달)
n03 → 규정에서 찾을 수 없습니다. (유사도 임계값 미달)
q01 → 학사과정의 수업연한은 **4년**입니다.  
다만, 건축대학의 수업연한은 **5년**입니다.

(출처: 명지대학교 학칙 제11조 제1항)
q02 → 인공지능·소프트웨어융합대학 학생의 졸업 이수학점은 **134학점**입니다.

(출처: 학칙 시행규칙(학사과정) 제108조 제1항 제1호)


> **주의**: 임계값을 너무 높이면 정답 있는 질문(q19)까지 거부됨 → 실무에서는 평가용과 별도의 검증용 질문으로 정함

---
## 사례 분석 ③ Top-K 줄이기와 정확도 변화
- 현상: 정답은 위에 있지만(Hit=1, MRR=1) 쓸모없는 청크도 같이 들어감 (Precision 25%)
- 진단: 검색 알고리즘이 아니라 **몇 개를 넘기느냐(Top-K)** 의 문제
- 조치: K를 4 → 2로 줄이고 비교
- 주의: K를 줄이면 **Recall(특히 근거가 2개인 종합형)** 이 떨어질 수 있음 → 함께 확인

In [35]:
result_k2 = evaluate(dense_search, k=2)

topk = pd.DataFrame({"K=4": summary(result), "K=2": summary(result_k2)})
topk.loc[["precision", "recall", "mrr", "faithfulness", "correct", "ctx_chars", "total_ms"]].round(3)

  0%|          | 0/25 [00:00<?, ?it/s]

,K=4,K=2
precision,0.250,0.386
recall,0.909,0.705
mrr,0.761,0.705
faithfulness,0.960,0.920
correct,0.880,0.840
ctx_chars,1746.040,878.280
total_ms,4389.790,3587.819


- 종합형 질문은 K를 줄였을 때 영향이 더 큼

In [36]:
# 종합형 질문은 K 축소에 더 민감
pd.concat([result.assign(K=4), result_k2.assign(K=2)]).groupby(["type", "K"])[["recall", "correct"]].mean().round(2)

recall  correct
type K                 
사실형  2    0.70     0.80
     4    0.90     0.85
정답없음 2     NaN     1.00
     4     NaN     1.00
종합형  2    0.75     1.00
     4    1.00     1.00

> **결과 보기**: K=2로 줄이면 Precision 0.25 → 0.39, 글자 수 절반, 응답 시간 단축. 하지만 Recall 0.91 → 0.71, Correctness 0.88 → 0.84로 하락 → **무조건 줄이는 게 답이 아님**

---
## 청킹 전략 비교 — 최적 청킹이 정말 좋은가?
- 같은 문제지, 같은 임베딩 모델로 **자르는 방식만 바꿔서** 검색 점수 비교 (LLM 호출 없음 → 비용 없음)
  - 고정 크기: 글자 수로만 자름
  - 재귀(조 경계): 조 경계 우선, 주석·머리말 처리 없음
  - 최적(규정 맞춤): 주석 제거 + 조 경계 + 조 머리말 반복 ← 이 노트북의 기본 방식
- 실험용 저장소는 Milvus 대신 **메모리 벡터 저장소**(`InMemoryVectorStore`) 사용 → 같은 코사인 유사도라 검색 순위는 동일
- `make_search_fn(store)`: 저장소를 받아 **검색 함수를 만들어 주는** 함수 (저장소만 바꿔 끼우기 위해)
- 전략마다 임베딩에 수 분 소요 (T4 GPU 기준)

In [37]:
from langchain_core.vectorstores import InMemoryVectorStore

def build_memory_store(chunks):
    """실험용 임시 벡터 저장소 (메모리에 저장, 코사인 유사도) — 임베딩 진행률 표시"""
    embeddings.show_progress = True
    vs = InMemoryVectorStore(embeddings)
    vs.add_documents(chunks, ids=[c.metadata["chunk_id"] for c in chunks])
    embeddings.show_progress = False
    return vs

def make_search_fn(vector_store):
    """저장소 하나를 받아서 search(q, k) 모양의 검색 함수를 만들어 돌려줌"""
    def search(q, k):
        return vector_store.similarity_search(q, k=k)
    return search

def chunks_fixed(docs, size=500):
    """고정 크기 청킹: 글자 수로만 자름"""
    out = CharacterTextSplitter(separator="", chunk_size=size, chunk_overlap=0).split_documents(docs)
    for i, c in enumerate(out):
        c.page_content = f"[{c.metadata['title']}] {c.page_content.strip()}"
        c.metadata.update(article="", chunk_id=f"fixed-{i}")
    return out

def chunks_recursive(docs, size=500, overlap=50):
    """재귀 청킹: 조 경계 우선 (주석 제거·머리말 반복 없음)"""
    out = RecursiveCharacterTextSplitter(separators=["\n\n", ". ", " ", ""], chunk_size=size,
                                         chunk_overlap=overlap, keep_separator="end").split_documents(docs)
    for i, c in enumerate(out):
        c.page_content = f"[{c.metadata['title']}] {c.page_content.strip()}"
        c.metadata.update(article="", chunk_id=f"recursive-{i}")
    return out

strategy_chunks = {
    "고정 크기": chunks_fixed(docs),
    "재귀(조 경계)": chunks_recursive(docs),
    "최적(규정 맞춤)": chunks,
}
strategy_stores = {}
for name, cs in strategy_chunks.items():
    print(f"[{name}] 청크 {len(cs)}개 임베딩")
    strategy_stores[name] = build_memory_store(cs)

strategy = {}
for name, vs in strategy_stores.items():
    scores = eval_retrieval(make_search_fn(vs), k=4)      # 전략마다 검색 점수표
    strategy[name] = scores[METRICS].mean()               # 평균
pd.DataFrame(strategy).T.round(3)

[고정 크기] 청크 3400개 임베딩


Batches:   0%|          | 0/107 [00:00<?, ?it/s]

[재귀(조 경계)] 청크 4540개 임베딩


Batches:   0%|          | 0/142 [00:00<?, ?it/s]

[최적(규정 맞춤)] 청크 4150개 임베딩


Batches:   0%|          | 0/130 [00:00<?, ?it/s]

,hit,precision,recall,mrr,ndcg,ctx_precision
고정 크기,0.955,0.239,0.909,0.663,0.707,0.663
재귀(조 경계),0.864,0.227,0.841,0.742,0.755,0.742
최적(규정 맞춤),0.909,0.250,0.909,0.761,0.793,0.750


> **결과 보기**: 최적(규정 맞춤)이 MRR 0.761 · NDCG 0.793으로 가장 높음 → 정답을 **더 위에** 올려 줌. 고정 크기는 Hit은 높지만 순위(MRR)가 낮음

---
## 검색 설정 Grid Search — 청크 크기 × Top-K 조합 실험
- Grid Search: 가능한 조합을 **표처럼 전부** 시험해 보는 방법 (청크 크기 3가지 × K 3가지 = 9가지)
- 검색 지표만 사용 → LLM 비용 없이 빠르게 반복 가능
- **성능(속도·글자 수) vs 정확도(Hit·Recall·NDCG)** 의 균형점 찾기
- 청크 크기 500은 앞에서 만든 저장소를 다시 쓰고, 300·1000만 새로 임베딩

In [38]:
CHUNK_SIZES = [300, 500, 1000]
TOP_KS = [2, 4, 8]

# 청크 크기별 저장소 준비 (500은 재사용)
stores, n_chunks = {500: strategy_stores["최적(규정 맞춤)"]}, {500: len(chunks)}
for size in CHUNK_SIZES:
    if size not in stores:
        size_chunks = make_chunks(docs, size=size, overlap=size // 10)   # 겹침은 크기의 10%
        print(f"[청크 크기 {size}] 청크 {len(size_chunks)}개 임베딩")
        stores[size], n_chunks[size] = build_memory_store(size_chunks), len(size_chunks)

grid, per_q = [], {}
for size in CHUNK_SIZES:
    search = make_search_fn(stores[size])
    for k in TOP_KS:
        r = eval_retrieval(search, k=k)                                  # 9가지 조합 각각 채점
        grid.append({"chunk_size": size, "K": k, "청크 수": n_chunks[size], **r[METRICS].mean(),
                     "ctx_chars": r["ctx_chars"].mean(), "search_ms": r["search_ms"].mean()})
        if k == 4:
            per_q[size] = r.set_index("id")["hit"]                       # K=4일 때 문항별 적중 기록

pd.DataFrame(grid).round(3)

[청크 크기 300] 청크 7077개 임베딩


Batches:   0%|          | 0/222 [00:00<?, ?it/s]

[청크 크기 1000] 청크 2000개 임베딩


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

,chunk_size,K,청크 수,hit,precision,recall,mrr,ndcg,ctx_precision,ctx_chars,search_ms
0,300,2,7077,0.727,0.364,0.705,0.659,0.666,0.659,518.591,378.460
1,300,4,7077,0.818,0.205,0.773,0.686,0.701,0.686,1040.591,374.988
2,300,8,7077,0.955,0.125,0.932,0.705,0.757,0.703,2046.045,374.206
3,500,2,4150,0.727,0.386,0.705,0.705,0.693,0.705,860.136,232.210
4,500,4,4150,0.909,0.250,0.909,0.761,0.793,0.750,1713.273,228.180
5,500,8,4150,1.000,0.136,1.000,0.774,0.822,0.762,3322.545,231.045
6,1000,2,2000,0.636,0.318,0.614,0.568,0.575,0.568,1681.091,120.189
7,1000,4,2000,0.864,0.216,0.818,0.644,0.680,0.644,3433.636,122.263
8,1000,8,2000,0.909,0.114,0.864,0.653,0.697,0.653,6892.000,120.785


- 강의자료 형식: 청크 크기별 **문항 적중표** (K=4, ✓=정답 청크를 찾음)

In [39]:
# 강의자료 형식: 청크 크기별 질문 적중표 (K=4)
table = pd.DataFrame(per_q).T.replace({1: "✓", 0: "✗"})
table["정답률"] = pd.DataFrame(per_q).T.mean(axis=1).map("{:.0%}".format)
table

id,q01,q02,q03,q04,q05,q06,q07,q08,q09,q10,...,q14,q15,q16,q17,q18,q19,q20,q21,q22,정답률
300,✓,✗,✓,✓,✗,✓,✗,✓,✓,✓,...,✓,✓,✓,✓,✓,✓,✓,✓,✓,82%
500,✓,✗,✓,✓,✓,✓,✗,✓,✓,✓,...,✓,✓,✓,✓,✓,✓,✓,✓,✓,91%
1000,✓,✗,✓,✓,✓,✓,✓,✗,✓,✓,...,✓,✓,✓,✓,✓,✓,✓,✓,✓,86%


> **결과 보기**: 청크 500 · K=4가 정답률 91%로 균형이 좋음. 청크 1000은 글자 수가 2배인데 오히려 정확도가 낮음 → **크게 자른다고 좋은 게 아님**

---
## 실패 포인트(FP) 진단 — 틀린 문항은 어디서 잘못됐나?

| 영역 | 실패 포인트 | 이 실습에서의 판단 기준 |
|---|---|---|
| 검색 | FP1 콘텐츠 없음 | 정답 근거가 규정에 없음 / 정답없음 질문에 답해 버림 |
| 검색 | FP2 랭킹 누락 | 근거는 있는데 상위 K개 안에 못 들어옴 (Hit@K = 0) |
| 검색 | FP3 컨텍스트 미포함 | 근거를 일부만 가져옴 (Recall < 1) |
| 생성 | FP4 추출 실패 | 근거를 다 받았는데 틀림, 또는 Faithfulness 낮음(환각) |
| 생성 | FP5 잘못된 포맷 | 요구한 출처 표기 누락 |
| 생성 | FP6·FP7 구체성·완전성 | 근거도 충분하고 충실한데 틀림 → 너무 포괄적이거나 일부 누락 |

- 진단 순서: **응답부터 보고**(Phase 1) → 문제 있으면 **검색 지표**(Phase 2) → **문서 자체**(Phase 3)

In [40]:
# 문항마다 '근거 문구가 규정에 실제로 있는지' 미리 계산
in_corpus = {g["id"]: all(norm(ev) in doc_by_id.get(src, "") for src, ev in g["evidence"]) for g in golden}

def diagnose(row):
    """점수표 한 줄(문항 하나)을 보고 실패 포인트를 판정"""
    if row["type"] == "정답없음":                                   # 정답없음 문항: 모른다고 했나?
        return "OK" if row["correct"] else "FP1 콘텐츠 없음 → 환각 답변"
    if row["correct"] and row["faithfulness"] >= 0.8:              # Phase 1: 답변이 맞고 근거도 충실 → 형식만 확인
        return "OK" if row["format_ok"] else "FP5 잘못된 포맷"
    if not in_corpus[row["id"]]:                                   # Phase 3: 규정에 근거가 없음
        return "FP1 콘텐츠 없음"
    if row["hit"] == 0:                                            # Phase 2: 정답 청크를 못 찾음
        return "FP2 랭킹 누락"
    if row["recall"] < 1:                                          #          일부만 찾음
        return "FP3 컨텍스트 미포함"
    if row["faithfulness"] < 0.8:                                  # 검색은 됐는데 → 생성 문제(환각)
        return "FP4 추출 실패(환각)"
    return "FP6/FP7 구체성·완전성 부족"                            # 검색·근거 모두 OK인데 틀림

for name, df in [("K=4", result), ("K=2", result_k2)]:
    df["diagnosis"] = df.apply(diagnose, axis=1)                    # 모든 문항 진단
    print(f"[{name}]", df["diagnosis"].value_counts().to_dict())

result.loc[result["diagnosis"] != "OK", ["id", "diagnosis", "recall", "faithfulness", "reason"]]

[K=4] {'OK': 21, 'FP6/FP7 구체성·완전성 부족': 2, 'FP5 잘못된 포맷': 1, 'FP2 랭킹 누락': 1}
[K=2] {'OK': 20, 'FP2 랭킹 누락': 4, 'FP5 잘못된 포맷': 1}


,id,diagnosis,recall,faithfulness,reason
4,q05,FP5 잘못된 포맷,1.0,1.0,"응답이 정답의 핵심 사실인 '17학점'을 정확히 포함하고 있으며, 표현 차이만 있을..."
5,q06,FP6/FP7 구체성·완전성 부족,1.0,1.0,"정답은 '7학점 이내'라고만 명시했으나, 응답은 '학기당 7학점 이내'를 기본으로 ..."
6,q07,FP2 랭킹 누락,0.0,1.0,"정답은 원격수업이 총 개설 교과목 학점의 20%를 초과할 수 없다고 명시했으나, 응..."
10,q11,FP6/FP7 구체성·완전성 부족,1.0,1.0,정답은 '제적'으로 명시되어 있으나 응답은 '퇴학'으로 표현하여 핵심 사실이 일치하...


> **결과 보기**: K=4에서는 대부분 OK, 실패는 생성 쪽(FP5·FP6/7) 3건 + 검색 쪽(FP2) 1건. K=2로 줄이면 **FP2(랭킹 누락)가 4건**으로 늘어남 → K를 줄인 대가

---
## 종합 리포트
- 개발할 때: 문제지(Golden Set)로 **회귀 테스트** → 청킹·임베딩·K·프롬프트를 바꿀 때마다 **바꾸기 전과 후 점수 비교**
- **지표 하나만 보지 않기**: Precision=노이즈, Recall=누락, MRR/NDCG=순서, Faithfulness=환각, Correctness=최종 정답

In [41]:
s = summary(result)
print(f"""[명지대 규정 RAG 평가 리포트] Dense 검색, K=4, 문항 {len(result)}개
- Hit@4        {s['hit']:.0%} : 정답 근거 청크를 상위 4개 안에 가져온 질문 비율
- Precision@4  {s['precision']:.0%} : 전달한 청크 중 정답 근거 청크 비율 (나머지는 노이즈)
- Recall@4     {s['recall']:.0%} : 필요한 정답 근거 중 회수한 비율
- MRR          {s['mrr']:.2f} : 첫 정답 근거 청크의 평균 순위 역수
- NDCG@4       {s['ndcg']:.0%} : 이상적 순서 대비 순위 품질
- Faithfulness {s['faithfulness']:.2f} : 답변 주장 중 컨텍스트로 뒷받침되는 비율 (1 - 환각률)
- Relevancy    {s['relevancy']:.2f} : 답변이 질문 의도에 부합하는 정도
- Correctness  {s['correct']:.0%} : 정답과 일치한 답변 비율
- Negative Rejection {s['negative_rejection']:.0%} : 정답 없는 질문에 '모른다'고 답한 비율
- 평균 컨텍스트 {s['ctx_chars']:.0f}자 / 평균 응답 시간 {s['total_ms']:.0f}ms""")

[명지대 규정 RAG 평가 리포트] Dense 검색, K=4, 문항 25개
- Hit@4        91% : 정답 근거 청크를 상위 4개 안에 가져온 질문 비율
- Precision@4  25% : 전달한 청크 중 정답 근거 청크 비율 (나머지는 노이즈)
- Recall@4     91% : 필요한 정답 근거 중 회수한 비율
- MRR          0.76 : 첫 정답 근거 청크의 평균 순위 역수
- NDCG@4       79% : 이상적 순서 대비 순위 품질
- Faithfulness 0.96 : 답변 주장 중 컨텍스트로 뒷받침되는 비율 (1 - 환각률)
- Relevancy    0.73 : 답변이 질문 의도에 부합하는 정도
- Correctness  88% : 정답과 일치한 답변 비율
- Negative Rejection 100% : 정답 없는 질문에 '모른다'고 답한 비율
- 평균 컨텍스트 1746자 / 평균 응답 시간 4390ms


---
## 정리 & 직접 해보기

### 오늘 배운 것
1. RAG = **찾아보고(검색) 답하기(생성)**. LLM 혼자서는 모르는 규정(134학점)도 근거와 함께 정확히 답함
2. 데이터 준비가 성능을 좌우: 주석 제거 + 조 경계 청킹 + 조 머리말 반복 → MRR·NDCG 향상
3. 평가는 **검색 / 응답 두 축**으로 나눠서: 검색 지표 6개는 LangChain에 없으므로 직접 함수로 구현
4. 설정(K, 청크 크기)은 **정확도와 속도의 균형**으로 결정 → Golden Set으로 실험해서 정함

### 직접 해보기 과제
1. `golden`에 나만의 질문 2개(사실형 1, 정답없음 1)를 추가하고 `evaluate()`를 다시 실행해 보기
2. `RAG_PROMPT`의 규칙을 바꿔(예: 출처 표기 규칙 삭제) `format_ok`와 FP5가 어떻게 변하는지 확인
3. `retrieved = ["c3", "c1", "c9", "c7"]`처럼 순서만 바꿔서 MRR·NDCG·Context Precision이 어떻게 달라지는지 계산해 보기